# Churn Prediction — Exploratory Data Analysis

## Objective

The objective of this project is to predict, based on customer features, which customers are likely to churn from a telecom company. My initial assumption is that I would expect a relatively low churn rate, since this is a snapshot of current customers, and most people who were going to churn have likely already left. Given my assumption, I would expect minimizing false negatives to matter most: predicting a customer won't churn when they actually will is more costly to the business (a lost customer with no intervention) than predicting churn for someone who was going to stay (a wasted retention offer). 

In this notebook the objective is to understand the structure, distributions, and relationships in the Teleco Customer Churn dataset to inform cleaning and feature engineering decisions.

## Inputs
- `../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`
- Teleco Customer Churn dataset — 7043 Customers, 21 columns
- Column Description: 
    - **Customer info**: `customerID`, `gender`, `SeniorCitizen`, `Partner`, `Dependents`
    - **Account & billing**: `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`
    - **Services subscribed**: `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
    - **Target**: `Churn`

## Output


## 1.1 Setup and Imports

Importing core libraries for data manipulation, visualization, and analysis. Path constant is defined here to ensure consistent file references throughout the notebook. Warning suppression keeps the output clean.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore') # suppress outputs of warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)

RAW_DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

## 1.2 load dataset

Using pandas to read the dataset and varifying the csv has been properly read into the dataframe with `.shape` and `.head()`. 

In [3]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Comparing against what's shown on Kaggle, the shape `(7043, 21)` confirms I've successfully imported 7043 samples and 21 columns, including the target `Churn`. The `.head()` output shows the first 5 rows, and the column names and values match what's described on Kaggle, confirming the data loaded correctly.

## 1.3 Sanity Check

For the sanity check I will first check to ensure . 

In [ ]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

given that the `dtype` for `MonthlyCharges` is a float but `TotalCharges` is a `str`, this raises questions that there may be possible data entry errors causing the `dtype` to be str and not a float. 

In [12]:
converted = pd.to_numeric(df['TotalCharges'], errors = 'coerce')
mask = converted.isna()
mask.sum()
df[mask][['TotalCharges', 'tenure']]


,TotalCharges,tenure
488,,0
753,,0
936,,0
1082,,0
1340,,0
3331,,0
3826,,0
4380,,0
5218,,0
6670,,0


In [ ]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

no missing values in any of the columns

In [8]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000
